## Calculate Inter-Annotator Agreement between annotators

### patient id that were annotated by both annotators

In [1]:
import pandas as pd

In [ ]:
# expand df: each row is one annotation, with annotator info
all_annotation_df = pd.read_csv("./data/all_annotation.csv")

# compact df: with total error count etc
final_annotation_df = pd.read_csv("./data/all_data.csv")

# get patient ID which were annotated by both annotators
overlapping_ids = (
    all_annotation_df.groupby("patientid")["annotation_user"]
      .nunique()
      .loc[lambda x: x > 1]
      .index
      .tolist()
)

# ID for calculating cohen's kappa. 1044 needs to be removed. 
# print(overlapping_ids)

In [ ]:
# check if those patient id is in final_annotation_df. Some examples are excluded due to incomplete annotation or no generated letters. 
# If the is is in final_annotation_df, record that patient id

overlapping_example_id = set(overlapping_ids).intersection(final_annotation_df["patientid"])
overlapping_example_id


In [4]:
len(overlapping_example_id)

16

## Calculate Precision, Recall, F1

In [5]:
def compute_annotation_f1(df, annotator_a, annotator_b):
    tp = 0
    for _, row in df.iterrows():

            # only look at duplicated item (duplicate != nan)
            if pd.notna(row["duplicate"]):

                # the original item duplicate_id 
                duplicate_target = row["duplicate"]
                patient_id = row["patientid"]
                error_type = row["type"]
                
                # check whether duplicate links annotations
                # belonging to different annotators

                original = df[
                    (df["patientid"] == patient_id)
                    & (df["type"] == error_type)
                    & (df["duplicate_id"] == duplicate_target)
                ]
        
                original_annotator = original.iloc[0]["annotation_user"]

                if original_annotator != row["annotation_user"]:
                    tp += 1

    n_a = (df["annotation_user"] == annotator_a).sum()
    n_b = (df["annotation_user"] == annotator_b).sum()

    precision = tp / n_b if n_b else 0
    recall = tp / n_a if n_a else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    return {
        "n_a": n_a, 
        "n_b": n_b,
        "tp": tp,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
selected_df = all_annotation_df[
    (all_annotation_df["evaluated_letter"] == "GPT letter")
    & (all_annotation_df["patientid"].isin(overlapping_example_id))
]

result = compute_annotation_f1(selected_df, "student_email_A", "student_email_B")

result

{'n_a': 177,
 'n_b': 109,
 'tp': 35,
 'precision': 0.3211009174311927,
 'recall': 0.1977401129943503,
 'f1': 0.24475524475524474}

## Correlation




In [7]:
from scipy.stats import pearsonr, spearmanr

def correlation_between_annotator(
    df,
    annotator_a,
    annotator_b,
    error_type,
    overlapping_example_ids,
     only_important_error=True
):
    values_a = []
    values_b = []

    for patient_id in overlapping_example_ids:
        if only_important_error: 
            anno_a = (
                (df["annotation_user"] == annotator_a)
                & (df["type"] == error_type)
                & (df["patientid"] == patient_id)
                & (df["importance"] == "Important")
            ).sum()

            anno_b = (
                (df["annotation_user"] == annotator_b)
                & (df["type"] == error_type)
                & (df["patientid"] == patient_id)
                & (df["importance"] == "Important")
            ).sum()
        else: 
            anno_a = (
                (df["annotation_user"] == annotator_a)
                & (df["type"] == error_type)
                & (df["patientid"] == patient_id)
            ).sum()

            anno_b = (
                (df["annotation_user"] == annotator_b)
                & (df["type"] == error_type)
                & (df["patientid"] == patient_id)
            ).sum()
            

        values_a.append(anno_a)
        values_b.append(anno_b)

    corr, p_value = spearmanr(values_a, values_b)

    return round(corr, 3), "{:.2e}".format(p_value) #round(p_value, 4) 

In [ ]:
hallu_all, hallu_all_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","hallucination", overlapping_example_id, only_important_error=False)
hallu_important, hallu_important_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","hallucination", overlapping_example_id, only_important_error=True)
omission_all, omission_all_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","omission", overlapping_example_id, only_important_error=False)
omission_important, omission_important_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","omission", overlapping_example_id, only_important_error=True)
trivial_all, trivial_all_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","trivial", overlapping_example_id, only_important_error=False)
trivial_important, trivial_important_p_value = correlation_between_annotator(selected_df, "student_email_A", "student_email_B","trivial", overlapping_example_id, only_important_error=True)

In [9]:
a = pd.DataFrame({"hallu_all": (hallu_all, hallu_all_p_value),
 "hallu_important" : (hallu_important, hallu_important_p_value),
 "omi_all": (omission_all, omission_all_p_value),
 "omi_important": (omission_important, omission_important_p_value), 
 "trivial_all": (trivial_all, trivial_all_p_value), 
 "trivial_important": (trivial_important, trivial_important_p_value)})

In [10]:
print(a)

  hallu_all hallu_important   omi_all omi_important trivial_all  \
0     0.429           0.477     0.597           0.7       0.802   
1  9.71e-02        6.16e-02  1.46e-02      2.51e-03    1.88e-04   

  trivial_important  
0             0.846  
1          3.61e-05  


## plot the error in barplot

In [ ]:
# mimic data 
mimic_data = pd.read_csv("./data/mimic-hallu.csv")

In [12]:
import numpy as np

def summarize(x):
    return {
        "mean": round(np.mean(x),3),
        "median": round(np.median(x)),
        "std": round(np.std(x)),
        "min": round(np.min(x)),
        "max": round(np.max(x)),
        "iqr": f"{round(np.percentile(x, 25))} - {round(np.percentile(x, 75))}",
        "zero_%": round(np.mean(x == 0) * 100)
    }

summary = {
    "hallucination all": summarize(final_annotation_df["gpt_all_hallu_count"]),
    "hallucination important": summarize(final_annotation_df["gpt_important_hallu_count"]),
    "omission all": summarize(final_annotation_df["gpt_all_omission_count"]),
    "omission important": summarize(final_annotation_df["gpt_important_omission_count"]),
    "trivial all": summarize(final_annotation_df["gpt_all_trivial_count"]),
    "trivial important": summarize(final_annotation_df["gpt_important_trivial_count"]),
    "MIMIC hallucination all": summarize(mimic_data["hallu_count"])
    
    
    
}

summary = pd.DataFrame(summary)


In [13]:
summary

,hallucination all,hallucination important,omission all,omission important,trivial all,trivial important,MIMIC hallucination all
mean,0.938,0.732,4.01,2.948,2.412,1.722,1.14
median,1,0,2,1,2,1,0
std,1,1,5,4,2,2,2
min,0,0,0,0,0,0,0
max,11,11,22,22,9,9,7
iqr,0 - 1,0 - 1,0 - 6,0 - 4,1 - 4,0 - 3,0 - 1
zero_%,48,57,26,37,20,34,50


In [55]:
summary.to_csv("annotation_summary.csv")